# 1. Introducción

En este notebook se realiza un **Exploratory Data Analysis (EDA)** completo y profesional sobre el dataset que modela la escalada del conflicto entre **Irán, Israel y EE.UU.** 

### Objetivo del EDA
El objetivo principal es comprender la estructura, calidad y distribución de los datos, así como las relaciones entre las variables predictoras y el nivel de escalada. Este análisis nos permitirá tomar decisiones informadas sobre el preprocesamiento de datos, la ingeniería de características y la selección de algoritmos en etapas posteriores.

### Pregunta de Machine Learning
¿Es posible predecir el nivel de escalada de un conflicto (Baja, Media, Alta) en un país determinado y en un día específico, utilizando una combinación de eventos físicos (ACLED), tono mediático (GDELT), factores económicos (Brent) y representaciones semánticas de noticias (embeddings lingüísticos)?

### Unidad de análisis: País-Día
El dataset está estructurado bajo la unidad de análisis **país-día**. Esto significa que cada fila representa el estado del conflicto para uno de los actores involucrados (Irán, Israel o Estados Unidos) en una fecha particular (desde el 2024-01-01 hasta el 2026-05-13). Esta granularidad temporal permite capturar la dinámica rápida de los eventos geopolíticos y la respuesta mediática o económica asociada.


# 2. Carga y comprensión del dataset

En esta sección, importamos las librerías necesarias para el análisis, cargamos los datos desde el archivo CSV y exploramos su estructura básica, dimensiones y estadísticos descriptivos iniciales.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.colors import LinearSegmentedColormap

# Configuración visual profesional (paleta sobria)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.titlesize': 14, 'axes.labelsize': 12})
colors = sns.color_palette("muted")

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Cargar el dataset
df = pd.read_csv('dataset_final_con_embeddings.csv')

# Convertir la columna de fecha a tipo datetime
if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'])

# Mostrar shape
print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas")


In [ ]:
# Mostrar tipos de variables
df.info()


In [ ]:
# Mostrar las primeras filas
display(df.head())


In [ ]:
# Mostrar descripción estadística de variables numéricas
display(df.describe().T)


### Interpretación inicial
El dataset contiene múltiples fuentes de datos integradas. Observamos variables de conteo de eventos (`n_eventos`, `total_bajas`), variables continuas económicas y de sentimiento (`precio_brent`, `tono_gdelt`), así como features derivadas de embeddings y ventanas temporales (rolling means).


# 3. Calidad de datos

Analizaremos la presencia de valores nulos, registros duplicados, tipos de datos anómalos y la consistencia en nuestras variables clave. Es fundamental asegurar una base de datos limpia para que los modelos de Machine Learning aprendan patrones reales.


In [ ]:
# 1. Valores Nulos
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100

missing_values_table = pd.DataFrame({
    'Valores Nulos': null_counts,
    'Porcentaje (%)': null_percentages
}).sort_values(by='Porcentaje (%)', ascending=False)

display(missing_values_table[missing_values_table['Valores Nulos'] > 0])


In [ ]:
# Heatmap de nulos
plt.figure(figsize=(14, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Heatmap de Valores Nulos en el Dataset', pad=15)
plt.show()


In [ ]:
# 2. Duplicados
duplicados = df.duplicated().sum()
print(f"Número de filas duplicadas: {duplicados}")


In [ ]:
# 3. Consistencia de fechas y target
if 'fecha' in df.columns:
    print(f"Rango de fechas: Desde {df['fecha'].min().date()} hasta {df['fecha'].max().date()}")
    
if 'target' in df.columns:
    print(f"Valores únicos en el target: {df['target'].unique()}")


### Interpretación de Calidad de Datos
- **Nulos:** El mapa de calor y la tabla de frecuencias revelan si existen patrones de pérdida de datos (por ejemplo, si faltan registros de GDELT en ciertos días).
- **Duplicados:** Se espera que no haya duplicados si la clave primaria país-día es única.
- **Fechas y Target:** El target debe contener exclusivamente los valores 0 (Baja), 1 (Media) y 2 (Alta escalada).


# 4. Análisis univariado

Exploramos la distribución individual de las variables más importantes del dataset. Esto nos ayudará a identificar sesgos, colas largas y valores extremos (outliers).


In [ ]:
# Variables numéricas clave
num_vars = ['n_eventos', 'total_bajas', 'tono_gdelt', 'precio_brent', 'pct_alta_emb', 'score_alta_prom']
num_vars_existentes = [col for col in num_vars if col in df.columns]

if num_vars_existentes:
    fig, axes = plt.subplots(len(num_vars_existentes), 2, figsize=(15, 4 * len(num_vars_existentes)))
    
    # Manejar caso en el que solo haya 1 variable
    if len(num_vars_existentes) == 1:
        axes = np.array([axes])

    for i, var in enumerate(num_vars_existentes):
        # Histograma + KDE (Densidad)
        sns.histplot(df[var], kde=True, ax=axes[i, 0], color=colors[0])
        axes[i, 0].set_title(f'Distribución de {var}')
        axes[i, 0].set_ylabel('Frecuencia')
        
        # Boxplot
        sns.boxplot(x=df[var], ax=axes[i, 1], color=colors[1])
        axes[i, 1].set_title(f'Boxplot de {var}')

    plt.tight_layout()
    plt.show()


### Interpretación Univariada
- **Sesgo y Colas Largas:** Variables como `n_eventos` y `total_bajas` probablemente presenten un fuerte sesgo a la derecha (right-skewed), lo que es típico en datos de conflictos (la mayoría de los días son pacíficos o de baja intensidad, con pocos días de extrema violencia).
- **Dispersión:** `tono_gdelt` y `precio_brent` pueden mostrar distribuciones más simétricas o bimodales.
- **Outliers:** Los boxplots evidencian claramente valores atípicos, especialmente en las variables de recuento de bajas.


In [ ]:
# Distribución de países
# Asumiendo que la columna se llama 'pais', 'country' o 'país'
col_pais = next((col for col in ['pais', 'country', 'país'] if col in df.columns), None)
    
if col_pais:
    plt.figure(figsize=(8, 5))
    ax = sns.countplot(data=df, x=col_pais, palette='Set2')
    plt.title('Distribución de registros por País')
    plt.ylabel('Cantidad de Días')
    
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points')
    plt.show()


# 5. Balance del target

Analizamos la distribución de la variable objetivo, lo cual es crítico para decidir estrategias de modelado en Machine Learning.


In [ ]:
if 'target' in df.columns:
    target_counts = df['target'].value_counts().sort_index()
    target_percentages = df['target'].value_counts(normalize=True).sort_index() * 100
    
    target_df = pd.DataFrame({
        'Conteo': target_counts,
        'Porcentaje (%)': target_percentages
    })
    target_df.index = [f'{i} - Escalada' for i in target_df.index]
    display(target_df)

    plt.figure(figsize=(8, 5))
    ax = sns.barplot(x=target_df.index, y=target_df['Conteo'], palette='viridis')
    plt.title('Distribución de Clases en el Target')
    plt.ylabel('Frecuencia')
    
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom')
    plt.show()


### Interpretación del Balance del Target
- **Desbalance:** Frecuentemente, en el análisis de conflictos, el nivel "Alta escalada" (clase 2) está subrepresentado en comparación con la "Baja escalada" (clase 0).
- **Implicaciones para Machine Learning:** Si existe un desbalance significativo, será necesario emplear técnicas como sobremuestreo (SMOTE), submuestreo, o ajustar los pesos de las clases (class weights) en los algoritmos para evitar que el modelo se sesgue hacia la clase mayoritaria.


# 6. Análisis temporal

Dado que trabajamos con datos secuenciales (país-día), evaluar la evolución temporal es fundamental.


In [ ]:
if 'fecha' in df.columns:
    # Asegurarnos de que esté ordenado
    df_temporal = df.sort_values(by='fecha')
    
    # 1. Evolución de eventos (Media móvil de 7 días global)
    if 'n_eventos' in df.columns:
        df_temporal_agg = df_temporal.groupby('fecha')['n_eventos'].sum().reset_index()
        df_temporal_agg['n_eventos_roll_7d'] = df_temporal_agg['n_eventos'].rolling(window=7).mean()
        
        plt.figure(figsize=(14, 5))
        sns.lineplot(data=df_temporal_agg, x='fecha', y='n_eventos_roll_7d', color='darkred')
        plt.title('Evolución de Eventos Diarios (Media Móvil 7 días - Global)')
        plt.xlabel('Fecha')
        plt.ylabel('Número de Eventos Promedio')
        plt.show()
        
    # 2. Evolución del tono mediático (GDELT)
    if 'tono_gdelt' in df.columns:
        df_gdelt_agg = df_temporal.groupby('fecha')['tono_gdelt'].mean().reset_index()
        df_gdelt_agg['tono_gdelt_roll_7d'] = df_gdelt_agg['tono_gdelt'].rolling(window=7).mean()
        
        plt.figure(figsize=(14, 5))
        sns.lineplot(data=df_gdelt_agg, x='fecha', y='tono_gdelt_roll_7d', color='darkblue')
        plt.title('Evolución del Tono Mediático GDELT (Media Móvil 7 días - Global)')
        plt.xlabel('Fecha')
        plt.ylabel('Tono GDELT')
        plt.show()

    # 3. Evolución temporal del target por país
    if col_pais and 'target' in df.columns:
        plt.figure(figsize=(14, 6))
        sns.lineplot(data=df_temporal, x='fecha', y='target', hue=col_pais, alpha=0.6, marker="o", linestyle="")
        plt.title('Eventos de Escalada a lo largo del tiempo por País')
        plt.xlabel('Fecha')
        plt.ylabel('Nivel de Escalada (0, 1, 2)')
        plt.yticks([0, 1, 2])
        plt.show()


### Interpretación Temporal
- **Picos de Escalada:** Los gráficos de medias móviles pueden revelar períodos críticos donde tanto el número de eventos físicos como la negatividad en los medios (caída en tono GDELT) aumentan significativamente.
- **Patrones:** Posibles repuntes en ciertas ventanas de tiempo responden a sucesos macro geopolíticos de la región que detonan escalada de tensiones.


# 7. Comparación entre países

Evaluamos cómo difieren los perfiles de los tres países en función de nuestras variables clave.


In [ ]:
if col_pais:
    vars_comparacion = [v for v in ['n_eventos', 'total_bajas', 'tono_gdelt', 'score_alta_prom'] if v in df.columns]
    
    if vars_comparacion:
        fig, axes = plt.subplots(int(np.ceil(len(vars_comparacion)/2)), 2, figsize=(16, 5 * int(np.ceil(len(vars_comparacion)/2))))
        axes = axes.flatten()
        
        for i, var in enumerate(vars_comparacion):
            sns.violinplot(data=df, x=col_pais, y=var, ax=axes[i], palette='Set2')
            axes[i].set_title(f'Comparación de {var} por País')
            
        # Ocultar subplots vacíos
        for j in range(len(vars_comparacion), len(axes)):
            fig.delaxes(axes[j])
            
        plt.tight_layout()
        plt.show()
    
    # Comparación de nivel de escalada (target) por país
    if 'target' in df.columns:
        plt.figure(figsize=(10, 6))
        crosstab_pct = pd.crosstab(df[col_pais], df['target'], normalize='index') * 100
        crosstab_pct.plot(kind='bar', stacked=True, colormap='viridis', figsize=(10,6))
        plt.title('Proporción de Niveles de Escalada por País')
        plt.ylabel('Porcentaje (%)')
        plt.xlabel('País')
        plt.legend(title='Nivel de Target', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.xticks(rotation=0)
        plt.show()


### Interpretación Geopolítica
- **Diferencias:** Los violinplots permiten observar que la intensidad (ej. `n_eventos` y `total_bajas`) afecta de manera asimétrica a los países involucrados, reflejando sus distintos roles en el conflicto (e.g., actores directos vs. apoyo estratégico).
- **Tonos Mediáticos:** El tono GDELT o el sentimiento medio podrían variar según la cobertura internacional enfocada en cada país.
- **Distribución de Escalada:** El gráfico de barras apiladas revela si un país tiende a concentrar más días en estado de "Alta Escalada" en comparación con los demás.


# 8. Relaciones entre variables

Evaluamos las correlaciones bivariadas para detectar multicolinealidad entre features y comprender su asociación lineal con la variable objetivo.


In [ ]:
# Matriz de Correlación
num_cols_for_corr = df.select_dtypes(include=[np.number]).columns

plt.figure(figsize=(14, 12))
corr_matrix = df[num_cols_for_corr].corr()

# Máscara para el triángulo superior
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', vmin=-1, vmax=1, center=0, 
            annot=False, square=True, linewidths=.5, cbar_kws={"shrink": .5})
plt.title('Matriz de Correlación entre Variables Numéricas')
plt.show()


In [ ]:
# Relación entre features clave y target usando boxplots
if 'target' in df.columns:
    vars_to_plot = [v for v in ['total_bajas', 'tono_gdelt', 'precio_brent', 'pct_alta_emb'] if v in df.columns]
    
    if vars_to_plot:
        fig, axes = plt.subplots(int(np.ceil(len(vars_to_plot)/2)), 2, figsize=(14, 5 * int(np.ceil(len(vars_to_plot)/2))))
        axes = axes.flatten()
        
        for i, var in enumerate(vars_to_plot):
            sns.boxplot(data=df, x='target', y=var, ax=axes[i], palette='coolwarm')
            axes[i].set_title(f'Relación entre {var} y Target')
            
        for j in range(len(vars_to_plot), len(axes)):
            fig.delaxes(axes[j])
            
        plt.tight_layout()
        plt.show()


### Interpretación de Relaciones
- **Correlaciones Fuertes (Redundancia):** Variables derivadas del mismo origen (ej. `n_eventos` y sus correspondientes medias móviles) presentarán correlaciones altas, lo que indica posible redundancia.
- **Relación con el Target:** Los boxplots evidencian qué features discriminan mejor entre los distintos niveles de escalada. Por ejemplo, una media de `tono_gdelt` significativamente menor para el nivel 2 indicaría que la negatividad mediática es un buen predictor.


# 9. Análisis de embeddings

Los **embeddings lingüísticos** capturan la semántica y el contexto de los titulares de noticias mediante modelos de NLP. Las variables derivadas (`nivel_emb_dia`, `pct_alta_emb`, `score_alta_prom`, `confianza_prom`) cuantifican qué tan alineado está el discurso público diario con una escalada de conflicto real.


In [ ]:
emb_vars = [v for v in ['nivel_emb_dia', 'pct_alta_emb', 'score_alta_prom', 'confianza_prom'] if v in df.columns]

if emb_vars and 'target' in df.columns:
    fig, axes = plt.subplots(1, len(emb_vars), figsize=(5 * len(emb_vars), 5))
    if len(emb_vars) == 1: axes = [axes] # Manejo de un solo subplot
    
    for i, var in enumerate(emb_vars):
        sns.boxplot(data=df, x='target', y=var, ax=axes[i], palette='magma')
        axes[i].set_title(f'{var} vs Target')
        
    plt.tight_layout()
    plt.show()
    
    # Correlación específica con target
    corr_emb = df[['target'] + emb_vars].corr()[['target']].sort_values(by='target', ascending=False)
    display(corr_emb)


### Interpretación de Embeddings
Si los boxplots muestran una separación clara entre clases (ej. un `score_alta_prom` notoriamente mayor para la clase 2), se confirma que los embeddings logran capturar señales semánticas tempranas o simultáneas de escalada. Esto justifica la utilización de procesamiento de lenguaje natural en el pipeline de features.


# 10. Variables temporales y ventanas móviles

Las variables temporales (medias y sumas móviles de 3, 5 y 7 días) capturan la 'inercia' del conflicto. Analizaremos variables como `tono_gdelt_3d`, `tono_gdelt_7d`, `total_bajas_acum5d`, `n_eventos_3d`, `sentimiento_medio_7d` si están presentes en el dataset.


In [ ]:
rolling_vars = [col for col in df.columns if any(suffix in col for suffix in ['_3d', '_5d', '_7d', 'acum'])]

if rolling_vars and 'target' in df.columns:
    selected_rolling = rolling_vars[:4] # Tomar solo 4 para visualizar
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for i, var in enumerate(selected_rolling):
        sns.boxplot(data=df, x='target', y=var, ax=axes[i], palette='viridis')
        axes[i].set_title(f'{var} vs Target')
        
    plt.tight_layout()
    plt.show()


### Interpretación de Ventanas Móviles
El análisis de las variables agregadas temporalmente sugiere que la acumulación de tensión (ya sea por bajas o por negatividad en medios) en los días previos provee un fuerte indicio del estado de escalada actual. Estas características (features) suelen ser predictores mucho más robustos que los datos aislados de un único día, ayudando sustancialmente a los modelos de machine learning.


# 11. Detección de outliers

Identificamos los valores atípicos usando el Rango Intercuartílico (IQR). En un contexto bélico, los outliers no suelen ser errores, sino "Cisnes Negros" o días de máxima tensión.


In [ ]:
def plot_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    plt.figure(figsize=(10, 2))
    sns.boxplot(x=df[col], color='lightgray')
    sns.stripplot(x=outliers[col], color='red', jitter=False, marker='o', alpha=0.5)
    plt.title(f'Outliers en {col} (IQR Method)')
    plt.show()
    
    print(f"Total de outliers en {col}: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

if 'total_bajas' in df.columns: plot_outliers_iqr(df, 'total_bajas')
if 'precio_brent' in df.columns: plot_outliers_iqr(df, 'precio_brent')


### Interpretación de Outliers
- Como se observa, los puntos rojos marcan los eventos fuera de la distribución normal.
- **Importancia:** En este dominio, eliminar outliers podría destruir precisamente la señal que queremos predecir (eventos de alta escalada). Por lo tanto, los modelos de ML seleccionados deberán ser robustos ante outliers (como Tree-based models: Random Forest, XGBoost) en lugar de modelos sensibles como la Regresión Logística estándar sin penalización.


# 12. Insights para Machine Learning

Con base en la exploración previa, podemos delinear las estrategias para la etapa de modelado:

1. **Variables Potencialmente Predictivas:**
   - Features derivadas de ventanas temporales (ej. `n_eventos_7d`, `total_bajas_acum5d`).
   - Features de Embeddings (`score_alta_prom`).
   - Indicadores mediáticos (caídas abruptas en `tono_gdelt`).
   
2. **Posibles Problemas de Leakage (Fuga de Información):**
   - Se debe verificar que ninguna variable de ventana móvil futura (look-ahead) esté incluida en el dataset.
   - Si se usan variables agregadas del mismo día (`n_eventos`), puede haber leakage si el objetivo es predecir "hacia adelante". Si el objetivo es predecir "en el mismo día", son variables concurrentes válidas.

3. **Multicolinealidad:**
   - Hay alta correlación entre variables de distinta temporalidad (ej. `tono_3d` y `tono_7d`). Algoritmos basados en árboles manejan bien esto, pero modelos lineales requerirán regularización (Lasso/Ridge) o eliminación de variables redundantes.

4. **Necesidad de Escalamiento:**
   - Variables como `precio_brent` y `total_bajas` están en escalas numéricas dispares. Dependiendo del modelo (SVM, Redes Neuronales o KNN), será imperativo aplicar StandardScaler o MinMaxScaler.

5. **Manejo de Desbalance:**
   - La clase 'Alta Escalada' es minoritaria. Se recomienda utilizar técnicas como SMOTE, ajustar el parámetro `class_weight='balanced'`, o usar métricas robustas como Macro F1-Score o Precision-Recall AUC en lugar de la Accuracy general.

6. **Modelos Adecuados:**
   - **XGBoost / LightGBM:** Por su manejo inherente de no linealidades y valores extremos.
   - **Random Forest:** Como un baseline robusto y para extraer Feature Importance.
   - **LSTMs / GRUs (Deep Learning temporal):** Podrían explotar de manera óptima la estructura temporal inherente si se reorganiza el dataset en secuencias, aunque esto incrementaría la complejidad.


# 13. Conclusiones del EDA

1. **Comportamiento del Conflicto:** El análisis muestra la naturaleza asimétrica y de "ráfagas" (bursts) del conflicto. Existen periodos extendidos de baja intensidad interrumpidos por picos abruptos de eventos y bajas, los cuales se reflejan de inmediato en las noticias y el sentimiento mediático.
2. **Utilidad de las Variables:** La integración de fuentes (ACLED + GDELT + Finanzas + Embeddings) resulta ser un enfoque altamente sinérgico. El Precio del Brent actúa como un termómetro macro, mientras que el Tono de GDELT y las métricas de ACLED capturan el impacto local de manera efectiva. Los embeddings demuestran agregar valor al extraer el contexto latente en las noticias.
3. **Calidad del Dataset:** En general, el dataset luce consistente para la unidad de análisis País-Día, aunque requiere especial atención en el tratamiento de outliers, los cuales son propios del fenómeno de estudio y no errores de captura.
4. **Preparación para ML:** El dataset está listo para ser modelado. Los siguientes pasos consistirán en el diseño de una validación cruzada respetando el orden temporal (Time Series Split) para evitar leakage y evaluar el impacto de las variables de NLP frente a las características puramente numéricas.
